In [1]:
# CELL 1: Load the complete scheme JSON

import json
from pathlib import Path

INPUT_FILE = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json")

with open(INPUT_FILE, encoding="utf-8") as f:
    data = json.load(f)

schemes = data if isinstance(data, list) else data["schemes"]

print("Total schemes:", len(schemes))

Total schemes: 4428


In [2]:
# CELL 2: Create one parent document per scheme section

from langchain_core.documents import Document

sections = [
    "brief_description",
    "details",
    "eligibility",
    "benefits",
    "exclusions",
    "application_process",
    "documents_required"
]

parents = []
faqs = []

def to_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

for scheme in schemes:
    scheme_id = scheme["scheme_id"]

    for section in sections:
        text = to_text(scheme.get(section)).strip()

        if text:
            parents.append(Document(
                page_content=f"Scheme: {scheme.get('name', '')}\nSection: {section}\n{text}",
                metadata={
                    "parent_id": f"{scheme_id}_{section}",
                    "scheme_id": scheme_id,
                    "section": section,
                    "document_type": "scheme_section",
                    "categories": scheme.get("categories", [])
                }
            ))

    for i, faq in enumerate(scheme.get("faqs", []), 1):
        if not isinstance(faq, dict):
            continue

        question = to_text(faq.get("question")).strip()
        answer = to_text(faq.get("answer")).strip()

        if question and answer:
            faqs.append(Document(
                page_content=f"Question: {question}\nAnswer: {answer}",
                metadata={
                    "doc_id": f"{scheme_id}_FAQ_{i:03d}",
                    "scheme_id": scheme_id,
                    "section": "faq",
                    "document_type": "faq"
                }
            ))

print("Parent documents:", len(parents))
print("FAQ documents:", len(faqs))

Parent documents: 27305
FAQ documents: 48695


In [3]:
# CELL 3: Split long parent documents into searchable child chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50
)

children = []

for parent in parents:
    if len(parent.page_content) <= 800:
        child_docs = [parent]
    else:
        child_docs = splitter.split_documents([parent])

    for i, child in enumerate(child_docs, 1):
        child.metadata["parent_id"] = parent.metadata["parent_id"]
        child.metadata["child_id"] = (
            f"{parent.metadata['parent_id']}_CHILD_{i:03d}"
        )

        children.append(child)

# FAQs remain atomic
for faq in faqs:
    faq.metadata["child_id"] = faq.metadata["doc_id"]
    faq.metadata["parent_id"] = None
    children.append(faq)

print("Total child documents:", len(children))

Total child documents: 124029


In [4]:
# CELL 4: Export parent and child documents

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_documents(documents, filename):
    with open(OUTPUT_DIR / filename, "w", encoding="utf-8") as f:
        for doc in documents:
            record = {
                "text": doc.page_content,
                "metadata": doc.metadata
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_documents(parents, "parent_documents.jsonl")
save_documents(children, "child_documents.jsonl")

print("Parent and child documents saved successfully!")

Parent and child documents saved successfully!


In [8]:
# CELL 1: Analyze generated child chunks

import pandas as pd

chunk_df = pd.DataFrame([
    {
        "scheme_id": doc.metadata["scheme_id"],
        "parent_id": doc.metadata.get("parent_id"),
        "child_id": doc.metadata["child_id"],
        "section": doc.metadata["section"],
        # "tokens": len(encoder.encode(doc.page_content))
    }
    for doc in children
])

print("Total chunks:", len(chunk_df))
print("Unique schemes:", chunk_df["scheme_id"].nunique())
print("Unique parents:", chunk_df["parent_id"].nunique())
print("\nChunks by section:")
print(chunk_df["section"].value_counts())

Total chunks: 124029
Unique schemes: 4428
Unique parents: 27305

Chunks by section:
section
faq                    48695
application_process    26139
details                12354
eligibility            11168
benefits               11116
documents_required      8887
brief_description       4428
exclusions              1242
Name: count, dtype: int64


In [11]:
# CELL 2: Analyze chunk token distribution

# print(chunk_df["tokens"].describe(
#     percentiles=[0.50, 0.90, 0.95, 0.99]
# ).round(2))

# print("\nChunks above 350 tokens:",
    #   (chunk_df["tokens"] > 350).sum())

In [12]:
# CELL 3: Validate document identifiers

print("Duplicate child IDs:", chunk_df["child_id"].duplicated().sum())

print("Missing parent IDs:",
      chunk_df["parent_id"].isna().sum())

print("Missing scheme IDs:",
      chunk_df["scheme_id"].isna().sum())

Duplicate child IDs: 0
Missing parent IDs: 48695
Missing scheme IDs: 0


In [14]:
# CELL 1: Analyze actual character and token lengths of generated chunks

chunk_df["characters"] = [
    len(doc.page_content) for doc in children
]

print(chunk_df[["characters"]].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).round(2))

       characters
count   124029.00
mean       271.02
std        136.24
min          2.00
50%        270.00
90%        386.00
95%        507.00
99%        724.00
max       8946.00


In [15]:
# CELL 1: Inspect chunks exceeding the configured 350-character limit

oversized = [
    doc for doc in children
    if len(doc.page_content) > 350
]

print("Oversized chunks:", len(oversized))

for doc in sorted(
    oversized,
    key=lambda x: len(x.page_content),
    reverse=True
)[:5]:
    print("\nSection:", doc.metadata["section"])
    print("Scheme:", doc.metadata["scheme_id"])
    print("Characters:", len(doc.page_content))
    print("Content preview:", doc.page_content[:300])

Oversized chunks: 16329

Section: faq
Scheme: BxgKwJoB5uF80_MG4VUX
Characters: 8946
Content preview: Question: What are the general requirements and conditions (across all components)?
Answer: 1. Assistance will be applicable for all APEDA scheduled products.
2. The date of submission of the online application in APEDA shall be considered as the date of receipt of an application for grant of princi

Section: faq
Scheme: BxgKwJoB5uF80_MG4VUX
Characters: 3395
Content preview: Question: What is APEDA ?
Answer: The Agricultural and Processed Food Products Export Development Authority (APEDA) was
established by the Government of India under the Agricultural and Processed Food Products Export Development Authority Act, 1985. The primary objective of APEDA is to undertake the

Section: faq
Scheme: BxgKwJoB5uF80_MG4VUX
Characters: 3354
Content preview: Question: What are the Scheme Components ?
Answer: The Financial Assistance Scheme (FAS) is an export promotion scheme run by APEDA. The
scheme